<a href="https://colab.research.google.com/github/gqs24/undergradprojects/blob/main/Macro_Inflation_Tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
# Install the official Python wrapper for the FRED API
!pip install fredapi
# Install localtunnel using Node Package Manager to expose our local server to the web
!npm install localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇
up to date, audited 23 packages in 1s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠏

In [33]:
# ==========================================
# 1. IMPORT REQUIRED LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
import requests
from fredapi import Fred

# ==========================================
# 2. LIVE DATA INGESTION (FRED & SINGSTAT PIPELINES)
# ==========================================
# --- A. FRED API (Global Commodity Indices) ---
# Initialize the FRED client
fred = Fred(api_key='add97eb05104bd5bbd287892b4dfd681')

# Fetch Global Price of Brent Crude and Global Price of Food Index
oil_series = fred.get_series('POILBREUSDM')
food_series = fred.get_series('PFOODINDEXM')

# Combine into a single DataFrame and filter from Jan 2018 onwards
fred_df = pd.DataFrame({'Global_Oil_Index': oil_series, 'Global_Food_Index': food_series})
fred_df = fred_df.loc['2018-01-01':]
fred_df.index = pd.to_datetime(fred_df.index)

# --- B. SINGSTAT API (Singapore CPI - M213751) ---
singstat_url = "https://tablebuilder.singstat.gov.sg/api/table/tabledata/M213751"

# Headers are required to mimic a browser and bypass API bot-blockers
headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36',
    'Accept': 'application/json'
}

response = requests.get(singstat_url, headers=headers)
raw_json = response.json()

# Extract rows from M213751.json
records = raw_json['Data']['row']
cpi_data_list = []

# Target the 'All Items' series as an example
for row in records:
    # rowText matches the labels in M213751.json
    if row.get('rowText') == 'All Items':
        for col in row.get('columns', []):
            # Use 'key' and 'value' verbatim from M213751.json
            date_str = col.get('key')
            val_str = col.get('value')
            if date_str and val_str:
                cpi_data_list.append({
                    'Date': date_str,
                    'SG_CPI': float(val_str)
                })

# Convert to DataFrame
singstat_df = pd.DataFrame(cpi_data_list)
# Convert format (e.g., "1961 Jan") to datetime
singstat_df['Date'] = pd.to_datetime(singstat_df['Date'], format='%Y %b')
singstat_df.set_index('Date', inplace=True)
singstat_df.sort_index(inplace=True)

print("Data successfully ingested from M213751.json:")
print(singstat_df.head())

# --- C. MERGING THE PIPELINES ---
# Merge FRED and SingStat data on matching dates, dropping any incomplete rows
data = pd.merge(fred_df, singstat_df, left_index=True, right_index=True, how='inner').dropna()

# ==========================================
# 3. DATA CLEANING & STATIONARITY (ADF TEST)
# ==========================================
# Define a function to run the Augmented Dickey-Fuller test for stationarity
def adf_test(series):
    result = adfuller(series.dropna())
    return result[1] # Return the p-value

# Transform the raw index data into month-over-month percentage changes to achieve stationarity
df_diff = data.pct_change().dropna() * 100

print("ADF P-Values (First Differenced):")
print(f"Core CPI: {adf_test(df_diff['SG_CPI']):.4f}")

# ==========================================
# 4. BASELINE REGRESSION MODEL (OLS)
# ==========================================
# Y = Differenced SG Core CPI; X = Differenced Global Commodities
Y = df_diff['SG_CPI']
X = df_diff[['Global_Oil_Index', 'Global_Food_Index']]
# Add a constant (intercept) to the regression model
X = sm.add_constant(X)

# Fit the OLS model and print the statistical summary
model = sm.OLS(Y, X).fit()
print("\n--- OLS Regression Results ---")
print(model.summary())

Data successfully ingested from M213751.json:
            SG_CPI
Date              
1961-01-01  21.071
1961-02-01  21.094
1961-03-01  21.112
1961-04-01  20.770
1961-05-01  20.655
ADF P-Values (First Differenced):
Core CPI: 0.2728

--- OLS Regression Results ---
                            OLS Regression Results                            
Dep. Variable:                 SG_CPI   R-squared:                       0.031
Model:                            OLS   Adj. R-squared:                  0.011
Method:                 Least Squares   F-statistic:                     1.542
Date:                Sat, 27 Jun 2026   Prob (F-statistic):              0.219
Time:                        04:37:41   Log-Likelihood:                -59.819
No. Observations:                 100   AIC:                             125.6
Df Residuals:                      97   BIC:                             133.5
Df Model:                           2                                         
Covariance Type:           

In [34]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import statsmodels.api as sm
import requests
from fredapi import Fred

# 1. PAGE CONFIGURATION
st.set_page_config(page_title="Macro-Inflation Tracker", layout="wide")
st.title("📊 Singapore Macro-Inflation Tracker")

# 2. CACHED DATA LOADING
@st.cache_data
def get_data():
    # Ingest FRED Data
    fred = Fred(api_key='add97eb05104bd5bbd287892b4dfd681')
    oil = fred.get_series('POILBREUSDM')
    food = fred.get_series('PFOODINDEXM')

    # Ingest SingStat Data
    singstat_url = "https://tablebuilder.singstat.gov.sg/api/table/tabledata/M213751"
    headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)', 'Accept': 'application/json'}
    response = requests.get(singstat_url, headers=headers).json()

    records = response['Data']['row']
    cpi_data = []
    for row in records:
        if row.get('rowText') == 'All Items':
            for col in row.get('columns', []):
                if col.get('Key') and col.get('Value'):
                    cpi_data.append({'Date': col['Key'], 'SG_CPI': float(col['Value'])})

    df_cpi = pd.DataFrame(cpi_data)
    df_cpi['Date'] = pd.to_datetime(df_cpi['Date'], format='%Y %b')
    df_cpi.set_index('Date', inplace=True)

    # Merge and Difference
    data = pd.DataFrame({'Oil': oil, 'Food': food, 'CPI': df_cpi['SG_CPI']}).dropna()
    diff = data.pct_change().dropna() * 100

    # Run OLS Regression
    Y = diff['CPI']
    X = sm.add_constant(diff[['Oil', 'Food']])
    model = sm.OLS(Y, X).fit()

    return model.params['Oil'], model.params['Food']

# 3. DASHBOARD UI
beta_oil, beta_food = get_data()

st.sidebar.header("Global Shock Parameters")
oil_shock = st.sidebar.slider("Global Oil Shock (%)", -50.0, 100.0, 0.0)
food_shock = st.sidebar.slider("Global Food Shock (%)", -50.0, 100.0, 0.0)

# Forecast calculation
baseline_inflation = 2.5
forecast = baseline_inflation + (oil_shock * beta_oil) + (food_shock * beta_food)

# Visualization
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(12), y=[baseline_inflation]*12, name='Baseline', line=dict(dash='dash', color='gray')))
fig.add_trace(go.Scatter(x=np.arange(12), y=np.linspace(baseline_inflation, forecast, 12), name='Forecast', line=dict(width=4, color='red')))
fig.update_layout(title="Inflation Forecast", yaxis_title="CPI % Change", xaxis_title="Month", template="plotly_white")
st.plotly_chart(fig, use_container_width=True)

st.subheader("Policy Outlook")
st.write(f"Based on your inputs, the forecasted CPI change is **{forecast:.2f}%**.")
if forecast > 4.0:
    st.error("⚠️ Inflation Warning: Monitor potential cost-push pressures.")
else:
    st.success("✅ Outlook: Inflation remains within manageable parameters.")

Overwriting app.py
